# Multimodal Product Embedding and Vector DB Ingestion

**Objective:** Create meaningful vector representations (embeddings) for each product by combining text and image data. Then, upload these embeddings into a Pinecone vector database for fast similarity search.

**Note:** This notebook uses pre-trained models — we are not training a model from scratch but leveraging powerful existing models (CLIP) for feature extraction.


In [2]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
from PIL import Image
import requests
import ast
from tqdm.auto import tqdm # For progress bars
import pinecone
import os

# Check for GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cpu


## 1. Load Cleaned Data
We’ll load the pre-cleaned product dataset, ensure image and text fields are valid, and limit to a manageable subset for testing.


In [3]:
# Cell 2: Load and Clean Data
df = pd.read_csv('intern_data_ikarus.csv')

# Drop rows with missing essential data
df.dropna(subset=['title', 'description', 'images'], inplace=True)

# Safely parse the string representation of image lists
def parse_string_list(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

df['images_list'] = df['images'].apply(parse_string_list)

# Use only the first image for generating embeddings
df['first_image'] = df['images_list'].apply(lambda x: x[0] if x else None)
df.dropna(subset=['first_image'], inplace=True)

print(f"Loaded {len(df)} products to process.")

Loaded 159 products to process.


## 2. Initialize Embedding Model

We use the `CLIP-ViT-B-32` model from `SentenceTransformers` — capable of embedding both images and text into the same vector space.


In [4]:
# Cell 3: Initialize Model
# ❗ RECTIFIED: The model name MUST be the full repository name.
# This will now load the model from your local cache, which you downloaded
# using the 'download_model.py' script.
clip_model = SentenceTransformer('sentence-transformers/clip-ViT-B-32', device=device)
print("✅ CLIP model loaded successfully.")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ CLIP model loaded successfully.


In [6]:
# Cell 4: Generate Embeddings (Corrected)

def get_image_embedding(image_url, embedding_dim):
    """Downloads an image and computes its embedding."""
    try:
        image = Image.open(requests.get(image_url, stream=True, timeout=10).raw).convert("RGB")
        return clip_model.encode(image)
    except Exception as e:
        # Now correctly returns a zero vector of the proper shape on failure
        # This prevents the entire process from crashing due to one bad image URL
        return np.zeros(embedding_dim)

def get_text_embedding(text):
    """Computes the text embedding."""
    return clip_model.encode(text)

# Combine text fields for a richer representation
df['combined_text'] = df.apply(
    lambda row: f"Title: {row['title']}. Description: {row['description']}. Brand: {str(row['brand'])}. Material: {str(row['material'])}. Color: {str(row['color'])}.",
    axis=1
)

# Use a smaller sample for demonstration to speed up the process
sample_df = df.head(100).copy()

# Initialize tqdm for pandas apply
tqdm.pandas()

print("Generating text embeddings...")
sample_df['text_embedding'] = sample_df['combined_text'].progress_apply(get_text_embedding)

# ❗ FIX 1: Get the embedding dimension from a successful text embedding.
# This is a robust way to find the correct size for our zero vectors.
# For CLIP ViT-B/32, this will be 512.
image_embedding_dim = sample_df['text_embedding'].iloc[0].shape[0]

print("\nGenerating image embeddings...")
# ❗ FIX 2: Pass the correct dimension to the function so it can create zero vectors on failure.
sample_df['image_embedding'] = sample_df['first_image'].progress_apply(
    lambda url: get_image_embedding(url, image_embedding_dim)
)

# Concatenate embeddings to create a single multimodal vector
sample_df['multimodal_embedding'] = sample_df.apply(
    lambda row: np.concatenate([row['text_embedding'], row['image_embedding']]),
    axis=1
)

print("\nEmbeddings generated successfully.")
print("Shape of a text embedding:", sample_df['text_embedding'].iloc[0].shape)
print("Shape of an image embedding:", sample_df['image_embedding'].iloc[0].shape)
print("Shape of a multimodal embedding:", sample_df['multimodal_embedding'].iloc[0].shape)

Generating text embeddings...


  0%|          | 0/100 [00:00<?, ?it/s]


Generating image embeddings...


  0%|          | 0/100 [00:00<?, ?it/s]


Embeddings generated successfully.
Shape of a text embedding: (512,)
Shape of an image embedding: (512,)
Shape of a multimodal embedding: (1024,)


In [7]:
# Cell 5: Initialize Pinecone

import pinecone
import os

# --- WARNING: Hardcoding API keys is not recommended for production. ---
# It is better to use environment variables to keep your secrets safe.
PINECONE_API_KEY = "pcsk_4oLe2C_ymfJT5FEUGV6j3iJQXMXc6gWxXtxf4kFJhfeC9gqsv1B5kujMacaypCg2wWFPh"

# Initialize connection to Pinecone. For serverless, you don't need an environment parameter here.
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY)

index_name = 'product-recommender'
# Calculate the dimension from the shape of the first multimodal embedding vector
embedding_dim = sample_df['multimodal_embedding'].iloc[0].shape[0]

# Check if the index already exists
if index_name not in pc.list_indexes().names():
    print(f"Creating a new serverless index: {index_name}")
    # Create the index with a ServerlessSpec.
    # You can change the cloud and region to your preference.
    pc.create_index(
        name=index_name,
        dimension=embedding_dim,
        metric='cosine',  # 'cosine' is recommended for CLIP embeddings
        spec=pinecone.ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    print("Index created successfully.")
else:
    print(f"Index '{index_name}' already exists. Connecting to it.")

# Connect to the index
index = pc.Index(index_name)

# Optional: Print index stats to confirm connection
print("\nIndex Stats:")
print(index.describe_index_stats())

Creating a new serverless index: product-recommender
Index created successfully.

Index Stats:
{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [8]:
# Cell 6: Prepare and Upsert Data to Pinecone
# Prepare data in the format Pinecone expects: (id, vector, metadata)
vectors_to_upsert = []
for _, row in sample_df.iterrows():
    # Metadata should be serializable and not too large
    metadata = {
        'title': row['title'],
        'brand': str(row['brand']),
        'price': str(row['price']), # Store as string
        'image_url': row['first_image']
    }
    vectors_to_upsert.append({
        'id': row['uniq_id'], # The unique ID for each vector
        'values': row['multimodal_embedding'].tolist(), # Vector values
        'metadata': metadata # Associated metadata
    })

# Upsert data in batches for efficiency
batch_size = 100
print(f"Upserting {len(vectors_to_upsert)} vectors into Pinecone index '{index_name}'...")
for i in tqdm(range(0, len(vectors_to_upsert), batch_size)):
    i_end = min(i + batch_size, len(vectors_to_upsert))
    batch = vectors_to_upsert[i:i_end]
    index.upsert(vectors=batch)

print("\nUpsert complete!")
print(index.describe_index_stats())

Upserting 100 vectors into Pinecone index 'product-recommender'...


  0%|          | 0/1 [00:00<?, ?it/s]


Upsert complete!
{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}
